# Введение в анализ данных. Где живут данные?

Этот блокнот — **рабочий лист** к занятию по open data, источникам данных и метаданным.

> Центральная идея пары: аналитик начинается не с графика и не с модели, а с умения **выбрать пригодные данные**.

## Что должно получиться к концу занятия
- вы различаете **dataset / портал / API / формат файла**;
- находите **3 потенциальных источника** по своей теме;
- читаете базовые **метаданные**;
- оформляете **паспорт датасета**;
- понимаете, где LLM помогает, а где проверка должна быть ручной.


## Как работать с этим ноутбуком

У занятия есть **две дорожки**:

### Обязательная для всех
1. посмотреть демонстрационные примеры;
2. найти 3 источника по своей теме;
3. сравнить их;
4. выбрать 1;
5. оформить паспорт датасета.

### Optional code track
Если вам комфортно в Colab, вы можете:
- посмотреть структуру CSV;
- посмотреть структуру JSON;
- сделать один GET-запрос к открытому API;
- собрать черновик паспорта прямо в ноутбуке.

### No-code track
Если код не нужен, используйте:
- браузер;
- Google Dataset Search / Kaggle / data.gov.ru / тематические порталы;
- `lesson02_no_code_workbook.xlsx`;
- `lesson02_dataset_passport_builder.html`.


In [ ]:
import io
import json
from textwrap import dedent

import pandas as pd
import requests
from IPython.display import HTML, Markdown, display

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 120)


In [ ]:
def card(title, body, color="#274c77", bg="#eef4fa"):
    html = f'''
    <div style="border:1px solid #d8e0ea;border-radius:16px;padding:16px 18px;background:{bg};margin:10px 0;">
        <div style="font-weight:700;color:{color};font-size:18px;margin-bottom:8px;">{title}</div>
        <div style="line-height:1.5;color:#1f2937;">{body}</div>
    </div>
    '''
    display(HTML(html))

card(
    "Что важно помнить",
    "<ul>"
    "<li><b>Open data</b> — это не просто «что-то в интернете», а данные с понятным источником и условиями использования.</li>"
    "<li><b>Метаданные</b> — это сведения о данных: кто собрал, когда обновил, в каком формате, что означают поля.</li>"
    "<li><b>LLM</b> помогает читать и оформлять, но не заменяет проверку лицензии, даты обновления и структуры.</li>"
    "</ul>"
)


## 1. Самый простой пример: табличный набор (CSV)

Начнём с искусственной таблицы. Здесь удобно проговорить:
- что является **единицей наблюдения**;
- какие поля числовые, а какие категориальные;
- какая информация отсутствует, если смотреть только на сам файл.


In [ ]:
csv_text = '''city,year,population,avg_temp,source
Moscow,2023,13100000,6.1,city_statistics
Kazan,2023,1318000,5.8,city_statistics
Novosibirsk,2023,1634000,1.9,city_statistics
Ekaterinburg,2023,1544000,3.4,city_statistics
'''

df_csv = pd.read_csv(io.StringIO(csv_text))
df_csv


### Вопросы
1. Что здесь является **единицей наблюдения**?
2. Какие поля можно считать **числовыми**, а какие — **категориальными**?
3. Чего не хватает, чтобы считать этот датасет «готовым к работе»?


## 2. Структурированные данные: JSON

API и многие веб-сервисы отдают данные не как готовую таблицу, а как **JSON**.
Это удобно для машин, но аналитику нужно уметь:
- увидеть структуру;
- понять вложенность;
- решить, как превратить это в таблицу.


In [ ]:
json_text = dedent('''
{
  "dataset": {
    "title": "Urban Mobility Demo",
    "publisher": "Open Transport Lab",
    "updated_at": "2026-02-15",
    "license": "CC BY 4.0",
    "records": [
      {"city": "Moscow", "metro_daily_riders_mln": 6.3, "year": 2025},
      {"city": "Saint Petersburg", "metro_daily_riders_mln": 2.1, "year": 2025},
      {"city": "Kazan", "metro_daily_riders_mln": 0.35, "year": 2025}
    ]
  }
}
''')

payload = json.loads(json_text)
display(payload)


In [ ]:
records = payload["dataset"]["records"]
df_json = pd.DataFrame(records)
df_json


### Вопросы
1. Где в JSON живут **метаданные**?
2. Где находятся собственно **наблюдения**?
3. Если бы это был реальный API, какие поля вы бы обязательно проверили на странице документации?


## 3. Открытый API без авторизации

Ниже — пример запроса к открытому API Всемирного банка.
Это хороший пример того, что:
- API — это тоже источник данных;
- данные приходят как JSON;
- их нужно интерпретировать, а не просто скачать.

Индикатор в примере: **Population, total** (`SP.POP.TOTL`).


In [ ]:
url = "https://api.worldbank.org/v2/country/NLD/indicator/SP.POP.TOTL?format=json&per_page=5"
response = requests.get(url, timeout=30)
response.status_code


In [ ]:
api_data = response.json()
type(api_data), len(api_data)


In [ ]:
api_data[0]


In [ ]:
df_api = pd.DataFrame(api_data[1])[["countryiso3code", "date", "value", "unit", "obs_status"]]
df_api.head()


### Что здесь важно заметить
- первая часть ответа API часто содержит **служебные метаданные**;
- вторая часть — сами наблюдения;
- даже при хорошем API нужно отдельно читать:
  - что означает индикатор;
  - за какой период доступны данные;
  - откуда они берутся;
  - какие есть ограничения интерпретации.


## 4. Быстрый no-code маршрут

Если вы не хотите кодить, ваш сценарий может быть таким:

1. Открыть один из стартовых источников:
   - Kaggle Datasets — https://www.kaggle.com/datasets
   - Google Dataset Search — https://datasetsearch.research.google.com/
   - data.gov.ru — https://data.gov.ru/
   - World Bank Open Data — https://data.worldbank.org/
   - WHO GHO — https://www.who.int/data/gho

2. Найти **3 источника** по теме.

3. Для каждого зафиксировать:
   - publisher / владелец;
   - дату обновления;
   - лицензию;
   - формат;
   - краткое описание структуры;
   - возможный риск.

4. Выбрать **1 лучший источник** и оформить по нему паспорт.

Для этого можно использовать `lesson02_no_code_workbook.xlsx` или HTML-конструктор.


## 5. Шаблон сравнения 3 источников

Ниже — табличный шаблон, который можно заполнить прямо в ноутбуке или перенести в Google Sheets.


In [ ]:
comparison = pd.DataFrame([
    {
        "theme": "",
        "source_name": "",
        "url": "",
        "source_type": "",
        "publisher": "",
        "updated_at": "",
        "license": "",
        "format": "",
        "why_relevant": "",
        "main_risk": "",
    }
    for _ in range(3)
])
comparison


## 6. Интерактивный черновик паспорта датасета

Эту секцию можно использовать как mini-form прямо в Colab.  
В Colab поля с `#@param` превращаются в форму, а в обычном Jupyter эти строки просто остаются комментариями.


In [ ]:
dataset_title = "Название датасета"  #@param {type:"string"}
dataset_url = "https://example.org/dataset"  #@param {type:"string"}
dataset_type = "Портал открытых данных"  #@param ["Портал открытых данных", "Каталог датасетов", "API", "CSV / файл", "Другое"]
dataset_publisher = "Владелец / publisher"  #@param {type:"string"}
dataset_updated = "2026-03-01"  #@param {type:"string"}
dataset_license = "Проверить вручную на странице источника"  #@param {type:"string"}
dataset_format = "CSV"  #@param ["CSV", "JSON", "API / JSON", "XLSX", "XML", "Несколько форматов"]
dataset_unit = "Одна строка = ..."  #@param {type:"string"}
dataset_fields = "id, date, category, value"  #@param {type:"string"}
dataset_task = "Для какой аналитической задачи подходит?"  #@param {type:"string"}
dataset_risks = "Какие ограничения и риски вы видите?"  #@param {type:"string"}

passport_html = f'''
<div style="border:1px solid #d8e0ea;border-radius:18px;overflow:hidden;">
  <div style="background:linear-gradient(135deg,#274c77,#3d6a99);color:white;padding:16px 18px;">
    <div style="font-size:24px;font-weight:700;">{dataset_title}</div>
    <div style="opacity:.92;margin-top:6px;">{dataset_type} · {dataset_format}</div>
  </div>
  <div style="padding:16px 18px;background:white;">
    <p><b>Ссылка:</b> {dataset_url}</p>
    <p><b>Publisher:</b> {dataset_publisher}</p>
    <p><b>Дата обновления:</b> {dataset_updated}</p>
    <p><b>Лицензия:</b> {dataset_license}</p>
    <p><b>Единица наблюдения:</b> {dataset_unit}</p>
    <p><b>Ключевые поля:</b> {dataset_fields}</p>
    <p><b>Подходит для:</b> {dataset_task}</p>
    <p><b>Ограничения и риски:</b> {dataset_risks}</p>
  </div>
</div>
'''
display(HTML(passport_html))


## 7. Практическое задание на паре

### Базовый уровень
1. Выберите тему, связанную с вашей программой.
2. Найдите 3 источника.
3. Сравните их.
4. Выберите 1 лучший.
5. Заполните паспорт датасета.

### Уровень выше
Дополнительно:
- откройте sample JSON / CSV;
- попробуйте понять структуру полей;
- сформулируйте 2–3 гипотезы, которые можно проверить на этих данных.

### Вопрос, на который вы должны суметь ответить
> Почему вы выбрали именно этот датасет, а не два других?


## 8. Чек-лист перед мини-защитой

Проверьте, что вы можете ответить на 5 вопросов:

1. Кто владелец источника?
2. Когда данные обновлялись?
3. Что является единицей наблюдения?
4. Есть ли лицензия / условия использования?
5. Какой главный риск или limitation у набора?


## 9. Домашнее продолжение

Дома вы доводите этот же датасет до полноценного паспорта:
- уточняете описание полей;
- фиксируете способ импорта;
- формулируете 2–3 аналитические гипотезы;
- optional: делаете первый импорт данных в Colab или предпросмотр в таблице.

> Идея: на следующем занятии вы уже не ищете данные заново, а работаете со своим выбранным набором.
